## Load already trained models and evaluate them

In [27]:
import gymnasium as gym
import pandas as pd
from collections import deque
import highway_env
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import random
from tqdm import tqdm
from stable_baselines3 import DQN as SB3_DQN
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from shared_core_config import SHARED_CORE_ENV_ID, SHARED_CORE_CONFIG
import imageio
from IPython.display import Image
from models import CustomDQNAgent, CustomDDQNAgent

### Define useful functions

In [28]:
def make_env(render_mode="rgb_array"):
    env = gym.make(SHARED_CORE_ENV_ID, render_mode=render_mode)
    env.unwrapped.configure(SHARED_CORE_CONFIG)
    env.reset()
    env = Monitor(env)
    return env

In [29]:
def evaluate_agent(agent, num_episodes=50, seeds=[3, 333, 3333]):
    """
    Evaluates an agent on multiple seeds.
    Returns a list of (seed, metrics_dict, episode_rewards) tuples.
    """
    print(f"Starting evaluation: {num_episodes} episodes per seed.")
    all_seed_results = []

    for seed in seeds:
        env = make_env()
        episode_rewards = []
        episode_lengths = []
        episode_crashed = []   # True if ended by collision, False if timeout

        for episode in tqdm(range(num_episodes), desc=f"Evaluating Seed {seed}"):
            obs, info = env.reset(seed=seed + episode)
            done = False
            total_reward = 0
            steps = 0

            while not done:
                if hasattr(agent, "predict"):
                    action, _states = agent.predict(obs, deterministic=True)
                else:
                    q_values = agent.get_q(obs)
                    action = np.argmax(q_values)
                obs, reward, terminated, truncated, info = env.step(action)
                total_reward += reward
                steps += 1
                done = terminated or truncated

            episode_rewards.append(total_reward)
            episode_lengths.append(steps)
            episode_crashed.append(bool(terminated)) 

        env.close()

        # Compute metrics
        mean_reward    = np.mean(episode_rewards)
        std_reward     = np.std(episode_rewards)
        min_reward     = np.min(episode_rewards)
        max_reward     = np.max(episode_rewards)

        mean_steps     = np.mean(episode_lengths)
        std_steps      = np.std(episode_lengths)
        min_steps      = np.min(episode_lengths)
        max_steps      = np.max(episode_lengths)

        n_crashes      = sum(episode_crashed)
        crash_rate     = n_crashes / num_episodes * 100
        survival_rate  = 100 - crash_rate

        metrics = {
            "mean_reward":    mean_reward,
            "std_reward":     std_reward,
            "min_reward":     min_reward,
            "max_reward":     max_reward,
            "mean_steps":     mean_steps,
            "std_steps":      std_steps,
            "min_steps":      min_steps,
            "max_steps":      max_steps,
            "n_crashes":      n_crashes,
            "crash_rate":     crash_rate,
            "survival_rate":  survival_rate,
        }

        print(f"\n{'='*50}")
        print(f"Seed {seed} — {num_episodes} episodes")
        print(f"{'='*50}")
        print(f"  Reward     : {mean_reward:6.2f} ± {std_reward:.2f}  "
              f"[min: {min_reward:.2f}, max: {max_reward:.2f}]")
        print(f"  Steps      : {mean_steps:6.1f} ± {std_steps:.1f}  "
              f"[min: {min_steps}, max: {max_steps}]")
        print(f"  Crashes    : {n_crashes}/{num_episodes} "
              f"({crash_rate:.1f}% crash rate, {survival_rate:.1f}% survival)\n")

        all_seed_results.append((seed, metrics, episode_rewards))

    # Aggregate across seeds 
    all_means    = [m["mean_reward"]   for _, m, _ in all_seed_results]
    all_survivals= [m["survival_rate"] for _, m, _ in all_seed_results]
    all_steps    = [m["mean_steps"]    for _, m, _ in all_seed_results]

    print(f"\n{'='*50}")
    print(f"OVERALL ({len(seeds)} seeds)")
    print(f"{'='*50}")
    print(f"  Mean reward    : {np.mean(all_means):.2f} ± {np.std(all_means):.2f}  (std is across seeds)")
    print(f"  Survival rate  : {np.mean(all_survivals):.1f}%")
    print(f"  Mean steps     : {np.mean(all_steps):.1f}")

    return all_seed_results

In [30]:
def build_comparison_table(named_results):
    """
    named_results: list of (model_name, eval_results)
    eval_results: list of (seed, metrics_dict, episode_rewards)
    """
    rows = []
    for model_name, results in named_results:
        row = {"Model": model_name}
        seed_means, seed_survivals, seed_steps = [], [], []

        for seed, metrics, _ in results:
            row[f"Seed {seed}\nmean±std reward"] = (
                f"{metrics['mean_reward']:.2f} ± {metrics['std_reward']:.2f}"
            )
            row[f"Seed {seed}\nsurvival%"] = f"{metrics['survival_rate']:.0f}%"
            seed_means.append(metrics["mean_reward"])
            seed_survivals.append(metrics["survival_rate"])
            seed_steps.append(metrics["mean_steps"])

        row["Overall\nmean reward"] = f"{np.mean(seed_means):.2f} ± {np.std(seed_means):.2f}"
        row["Overall\nsurvival%"]   = f"{np.mean(seed_survivals):.1f}%"
        row["Overall\nmean steps"]  = f"{np.mean(seed_steps):.1f}"
        rows.append(row)

    return pd.DataFrame(rows)

### Build first environment

In [31]:
temp_env = make_env()
obs_space = temp_env.observation_space
act_space = temp_env.action_space
temp_env.close()
print(f"Obs space: {obs_space.shape} | Act space: {act_space.n}")

Obs space: (10, 5) | Act space: 5


### Load models

In [32]:
MODEL_DIR = "./models"

# SB3
sb3_model = SB3_DQN.load(f"{MODEL_DIR}/dqn_sb3_highway")

# DQN 500
dqn_500 = CustomDQNAgent(act_space, obs_space)
dqn_500.q_net.load_state_dict(torch.load(f"{MODEL_DIR}/500_trained_dqn_highway.pth", weights_only=True))
dqn_500.q_net.eval()

# DQN 2000
dqn_2000 = CustomDQNAgent(act_space, obs_space)
dqn_2000.q_net.load_state_dict(torch.load(f"{MODEL_DIR}/2000_trained_dqn_highway.pth", weights_only=True))
dqn_2000.q_net.eval()

# DDQN 500
ddqn_500 = CustomDDQNAgent(act_space, obs_space)
ddqn_500.q_net.load_state_dict(torch.load(f"{MODEL_DIR}/500_trained_ddqn_highway.pth", weights_only=True))
ddqn_500.q_net.eval()

print("All models loaded successfully")

All models loaded successfully


### Evaluate each model

In [33]:
EVAL_SEEDS = [3, 333, 3333]
NUM_EPISODES = 50

In [34]:
print("="*60)
print("SB3 Baseline")
print("="*60)
sb3_results = evaluate_agent(sb3_model, num_episodes=NUM_EPISODES, seeds=EVAL_SEEDS)
np.save("./models/eval_sb3.npy", sb3_results, allow_pickle=True)
sb3_results = np.load("./models/eval_sb3.npy", allow_pickle=True).tolist()

SB3 Baseline
Starting evaluation: 50 episodes per seed.


Evaluating Seed 3: 100%|██████████| 50/50 [07:43<00:00,  9.26s/it]



Seed 3 — 50 episodes
  Reward     :  12.57 ± 6.18  [min: 1.60, max: 21.65]
  Steps      :   16.0 ± 8.6  [min: 3, max: 30]
  Crashes    : 40/50 (80.0% crash rate, 20.0% survival)



Evaluating Seed 333: 100%|██████████| 50/50 [04:38<00:00,  5.57s/it]



Seed 333 — 50 episodes
  Reward     :  12.70 ± 6.05  [min: 2.40, max: 23.91]
  Steps      :   15.9 ± 8.4  [min: 4, max: 30]
  Crashes    : 40/50 (80.0% crash rate, 20.0% survival)



Evaluating Seed 3333:  44%|████▍     | 22/50 [02:24<03:03,  6.55s/it]


KeyboardInterrupt: 

In [ ]:
print("="*60)
print("DQN - 500 episodes")
print("="*60)
dqn_500_results = evaluate_agent(dqn_500, num_episodes=NUM_EPISODES, seeds=EVAL_SEEDS)
np.save("./models/eval_dqn_500.npy", dqn_500_results, allow_pickle=True)
dqn_500_results  = np.load("./models/eval_dqn_500.npy", allow_pickle=True).tolist()

In [ ]:
print("="*60)
print("DQN - 2000 episodes")
print("="*60)
dqn_2000_results = evaluate_agent(dqn_2000, num_episodes=NUM_EPISODES, seeds=EVAL_SEEDS)
np.save("./models/eval_dqn_2000.npy", dqn_2000_results, allow_pickle=True)
dqn_2000_results = np.load("./models/eval_dqn_2000.npy", allow_pickle=True).tolist()

In [ ]:
print("="*60)
print("DDQN - 500 episodes")
print("="*60)
ddqn_500_results = evaluate_agent(ddqn_500, num_episodes=NUM_EPISODES, seeds=EVAL_SEEDS)
np.save("./models/eval_ddqn_500.npy", ddqn_500_results, allow_pickle=True)
ddqn_500_results = np.load("./models/eval_ddqn_500.npy",  allow_pickle=True).tolist()

Summing up tables

In [ ]:
named_results = [
    ("SB3 Baseline", sb3_results),
    ("DQN (500 ep)", dqn_500_results),
    ("DQN (2000 ep)", dqn_2000_results),
    ("DDQN (500 ep)", ddqn_500_results),
]

df = build_comparison_table(named_results)
print("\n" + "="*80)
print("COMPARISON TABLE")
print("="*80)
print(df.to_markdown(index=False))